# AgeLens — Project Setup

This notebook initializes the complete **AgeLens/NHANES project structure** without overwriting existing work.

## Placement

Save this notebook here:

```text
nhanes/
└── notebooks/
    └── 00_setup_agelens.ipynb
```

Then open the notebook and run all cells.

The notebook searches upward for the folder named `nhanes`, so it works whether Jupyter starts inside `nhanes/`, `nhanes/notebooks/`, or another project subfolder.

## Documentation installation

The setup creates the complete governance and methodology tree, including:

```text
docs/
├── _incoming/
├── governance/
└── methodology/
    └── paper_reviews/
```

To let the notebook organize existing Markdown and Excel documents automatically, place them either:

- directly in the `nhanes/` project root; or
- inside `nhanes/docs/_incoming/`.

Known files are copied to their canonical locations. Existing canonical files are preserved.

## Safety rules

- Existing configuration files are not overwritten unless `OVERWRITE_CONFIG = True`.
- Existing support files are not overwritten unless `OVERWRITE_SUPPORT_FILES = True`.
- Existing authoritative documents are never overwritten by the document installer.
- The setup never creates blank substitutes for missing authoritative documents.
- Existing folders and files are preserved.
- The setup does not download NHANES data.
- The setup creates and configures a dedicated mortality-data folder.
- Mortality files are not parsed in this setup notebook.
- Open Core Evidence Gaps remain explicitly recorded.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import sys

import pandas as pd

PROJECT_FOLDER_NAME = "nhanes"
OVERWRITE_CONFIG = False
OVERWRITE_SUPPORT_FILES = False
AUTO_ORGANIZE_DOCUMENTS = True
REQUIRE_ALL_DOCUMENTS = False

print(f"Python: {sys.version.split()[0]}")
print(f"Current working directory: {Path.cwd().resolve()}")


Python: 3.13.14
Current working directory: <PROJECT_ROOT>\notebooks


## 1. Locate the project root

The code refuses to create a second nested `nhanes/nhanes/` project accidentally.


In [2]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'.\n"
        "Move this notebook into nhanes/notebooks/ or start Jupyter "
        "from inside the nhanes project folder."
    )


PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")


Project root: <PROJECT_ROOT>


## 2. Create the project directory structure

Raw NHANES laboratory files are separated by cycle. Public-use linked mortality files are stored separately because they use a fixed-width `.dat` format and require a dedicated parser.

The documentation tree separates authoritative governance records, methodology documents, and paper reviews. `docs/_incoming/` is a temporary staging area for documents that the setup notebook can organize automatically.

The `scripts/` directory stores reproducible auxiliary code, including BioAge and R `survey` validation scripts.


In [3]:
DIRECTORIES = [
    "configs",
    "data/raw/2015_2016",
    "data/raw/2017_2018",
    "data/raw/mortality/2019_public",
    "data/interim",
    "data/processed",
    "docs/_incoming",
    "docs/governance",
    "docs/methodology",
    "docs/methodology/paper_reviews",
    "logs",
    "notebooks",
    "results/figures",
    "results/tables",
    "scripts",
    "src/agelens",
    "src/agelens/io",
    "src/agelens/preprocessing",
    "src/agelens/phenotypic_age",
    "src/agelens/validation",
    "src/agelens/survey",
    "src/agelens/mortality",
    "tests",
]

created = []
already_present = []

for relative_path in DIRECTORIES:
    directory = PROJECT_ROOT / relative_path
    if directory.exists():
        already_present.append(relative_path)
    else:
        directory.mkdir(parents=True, exist_ok=False)
        created.append(relative_path)

print(f"Created {len(created)} directories.")
for item in created:
    print(f"  + {item}")

print(f"Already present: {len(already_present)}")


Created 1 directories.
  + docs/_incoming
Already present: 22


## 3. Organize governance and methodology documents

This step uses a canonical document manifest.

The installer:

- accepts common duplicate-download suffixes such as `(1)` or `(6)`;
- copies recognized files from the project root or `docs/_incoming/`;
- preserves existing canonical files;
- never creates blank authoritative documents;
- refuses ambiguous source matches;
- records the result in `logs/document_installation_audit.json`.

The unversioned `00_Research_Protocol.md` is intentionally not treated as a substitute for the governed `00_Research_Protocol_v1.0.md`.


In [4]:
from shutil import copy2
import re


DOCUMENT_MANIFEST = {
    "docs/governance/00_Research_Protocol_v1.0.md": [
        "00_Research_Protocol_v1.0.md",
    ],
    "docs/governance/Assumption_Register.md": [
        "Assumption_Register.md",
    ],
    "docs/governance/Decision_Log.md": [
        "Decision_Log.md",
    ],
    "docs/governance/Evidence_Gap_Register.md": [
        "Evidence_Gap_Register.md",
    ],
    "docs/governance/Evidence_Matrix.xlsx": [
        "Evidence_Matrix.xlsx",
    ],
    "docs/governance/Literature_Matrix.xlsx": [
        "Literature_Matrix.xlsx",
    ],
    "docs/methodology/Methodology.md": [
        "Methodology.md",
    ],
    "docs/methodology/NHANES_Harmonization_Report.md": [
        "NHANES_Harmonization_Report.md",
    ],
    "docs/methodology/Replication_Protocol.md": [
        "Replication_Protocol.md",
    ],
    "docs/methodology/Validation_Protocol.md": [
        "Validation_Protocol.md",
    ],
    "docs/methodology/Variable_Mapping_Table.xlsx": [
        "Variable_Mapping_Table.xlsx",
    ],
    "docs/methodology/paper_reviews/Paper_001_Levine2018.md": [
        "Paper_001_Levine2018.md",
    ],
    "docs/methodology/paper_reviews/Paper_002_BioAge.md": [
        "Paper_002_BioAge.md",
    ],
    "docs/methodology/paper_reviews/Paper_003_LiuEtAl2018.md": [
        "Paper_003_LiuEtAl2018.md",
    ],
    "docs/methodology/paper_reviews/Paper_004_SelvinEtAl2007.md": [
        "Paper_004_SelvinEtAl2007.md",
    ],
}


def remove_duplicate_download_suffix(
    filename: str,
) -> str:
    path = Path(filename)

    cleaned_stem = re.sub(
        r"\s*\(\d+\)$",
        "",
        path.stem,
    )

    return f"{cleaned_stem}{path.suffix}"


def discover_document_sources() -> list[Path]:
    incoming_root = PROJECT_ROOT / "docs" / "_incoming"

    candidates = []

    # Only inspect files placed deliberately in supported staging locations.
    for path in PROJECT_ROOT.iterdir():
        if path.is_file():
            candidates.append(path.resolve())

    if incoming_root.exists():
        candidates.extend(
            path.resolve()
            for path in incoming_root.rglob("*")
            if path.is_file()
        )

    return sorted(set(candidates))


def install_documents() -> list[dict[str, object]]:
    source_files = discover_document_sources()

    source_index: dict[str, list[Path]] = {}

    for source_path in source_files:
        cleaned_name = remove_duplicate_download_suffix(
            source_path.name
        ).lower()

        source_index.setdefault(
            cleaned_name,
            [],
        ).append(source_path)

    audit_records: list[dict[str, object]] = []

    for destination_relative, aliases in DOCUMENT_MANIFEST.items():
        destination = PROJECT_ROOT / destination_relative
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if destination.exists():
            audit_records.append(
                {
                    "destination": destination_relative,
                    "status": "preserved_existing",
                    "source": None,
                }
            )
            continue

        matching_sources = []

        for alias in aliases:
            matching_sources.extend(
                source_index.get(
                    alias.lower(),
                    [],
                )
            )

        matching_sources = sorted(
            set(matching_sources)
        )

        if len(matching_sources) > 1:
            raise RuntimeError(
                "Ambiguous document sources for "
                f"{destination_relative}: {matching_sources}"
            )

        if len(matching_sources) == 1:
            source_path = matching_sources[0]

            copy2(
                source_path,
                destination,
            )

            audit_records.append(
                {
                    "destination": destination_relative,
                    "status": "installed",
                    "source": str(
                        source_path.relative_to(
                            PROJECT_ROOT
                        )
                    ),
                }
            )
        else:
            audit_records.append(
                {
                    "destination": destination_relative,
                    "status": "missing_source",
                    "source": None,
                }
            )

    return audit_records


if AUTO_ORGANIZE_DOCUMENTS:
    document_audit = install_documents()
else:
    document_audit = [
        {
            "destination": destination,
            "status": (
                "preserved_existing"
                if (PROJECT_ROOT / destination).exists()
                else "not_checked"
            ),
            "source": None,
        }
        for destination in DOCUMENT_MANIFEST
    ]

document_audit_path = (
    PROJECT_ROOT
    / "logs"
    / "document_installation_audit.json"
)
document_audit_path.write_text(
    json.dumps(
        {
            "captured_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "auto_organize_documents": (
                AUTO_ORGANIZE_DOCUMENTS
            ),
            "require_all_documents": (
                REQUIRE_ALL_DOCUMENTS
            ),
            "records": document_audit,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

installed_count = sum(
    record["status"] == "installed"
    for record in document_audit
)
preserved_count = sum(
    record["status"] == "preserved_existing"
    for record in document_audit
)
missing_documents = [
    record["destination"]
    for record in document_audit
    if record["status"] == "missing_source"
]

print(
    "Document installation summary: "
    f"installed={installed_count}, "
    f"preserved={preserved_count}, "
    f"missing={len(missing_documents)}"
)

for record in document_audit:
    symbol = {
        "installed": "+",
        "preserved_existing": "=",
        "missing_source": "!",
        "not_checked": "?",
    }.get(record["status"], "-")

    print(
        f"  {symbol} {record['destination']} "
        f"[{record['status']}]"
    )

if missing_documents:
    print(
        "\nMissing document sources were not replaced "
        "with blank files."
    )
    print(
        "Place the files in the project root or "
        "docs/_incoming and rerun this cell:"
    )

    for document in missing_documents:
        print(f"  - {document}")

    if REQUIRE_ALL_DOCUMENTS:
        raise FileNotFoundError(
            "Required project documents are missing:\n"
            + "\n".join(missing_documents)
        )

print(
    f"Document audit written: {document_audit_path}"
)


Document installation summary: installed=0, preserved=15, missing=0
  = docs/governance/00_Research_Protocol_v1.0.md [preserved_existing]
  = docs/governance/Assumption_Register.md [preserved_existing]
  = docs/governance/Decision_Log.md [preserved_existing]
  = docs/governance/Evidence_Gap_Register.md [preserved_existing]
  = docs/governance/Evidence_Matrix.xlsx [preserved_existing]
  = docs/governance/Literature_Matrix.xlsx [preserved_existing]
  = docs/methodology/Methodology.md [preserved_existing]
  = docs/methodology/NHANES_Harmonization_Report.md [preserved_existing]
  = docs/methodology/Replication_Protocol.md [preserved_existing]
  = docs/methodology/Validation_Protocol.md [preserved_existing]
  = docs/methodology/Variable_Mapping_Table.xlsx [preserved_existing]
  = docs/methodology/paper_reviews/Paper_001_Levine2018.md [preserved_existing]
  = docs/methodology/paper_reviews/Paper_002_BioAge.md [preserved_existing]
  = docs/methodology/paper_reviews/Paper_003_LiuEtAl2018.md [p

## 4. Create the machine-readable AgeLens configuration

The configuration mirrors approved governance decisions but does not replace the Research Protocol, Decision Log, Evidence Gap Register, or Replication Protocol.

Because EG-004, EG-010, and EG-014 remain open Core gaps, final scientific results remain disabled.


In [5]:
CONFIG = {
    "project": {
        "name": "AgeLens",
        "project_root_folder": "nhanes",
        "software_version": "0.1.0",
        "config_schema_version": "1.1",
        "status": "pre-validation",
        "final_scientific_results_allowed": False
    },
    "governance": {
        "research_protocol": {
            "document_id": "AL-RP-001",
            "version": "1.0",
            "status": "Draft / Internal Review"
        },
        "decisions": {
            "formula": "D-001",
            "cycle_scope": "D-009",
            "hscrp_bridging": "D-003",
            "biopro_bridging": "D-004",
            "bridging_direction": "D-005",
            "missing_data": "D-006",
            "unit_conversions": "D-007",
            "bridging_equations": "D-008"
        },
        "open_core_evidence_gaps": [
            "EG-004",
            "EG-010",
            "EG-014"
        ]
    },
    "nhanes": {
        "cycles": {
            "2015_2016": {
                "suffix": "I",
                "demographics": "DEMO_I.XPT",
                "biochemistry": "BIOPRO_I.XPT",
                "fasting_glucose": "GLU_I.XPT",
                "hscrp": "HSCRP_I.XPT",
                "cbc": "CBC_I.XPT"
            },
            "2017_2018": {
                "suffix": "J",
                "demographics": "DEMO_J.XPT",
                "biochemistry": "BIOPRO_J.XPT",
                "fasting_glucose": "GLU_J.XPT",
                "hscrp": "HSCRP_J.XPT",
                "cbc": "CBC_J.XPT"
            }
        },
        "merge_key": "SEQN",
        "cycle_column": "NHANES_CYCLE",
        "age_variable": "RIDAGEYR",
        "age_topcode_value": 80,
        "age_topcode_flag": "age_topcoded",
        "survey_design": {
            "weight_variable": "WTSAF2YR",
            "pooled_weight_variable": "WTSAF4YR",
            "pooled_weight_formula": "WTSAF2YR / 2",
            "stratum_variable": "SDMVSTRA",
            "psu_variable": "SDMVPSU"
        }
    },
    "mortality": {
        "enabled": True,
        "release": "2019_public",
        "raw_directory": "data/raw/mortality/2019_public",
        "format": "fixed_width_dat",
        "parse_in_setup": False,
        "merge_key": "SEQN",
        "files": {
            "2015_2016": "NHANES_2015_2016_MORT_2019_PUBLIC.dat",
            "2017_2018": "NHANES_2017_2018_MORT_2019_PUBLIC.dat"
        },
        "required_parser_source": "official_NCHS_sample_program_or_data_dictionary",
        "status": "stored_but_not_ingested"
    },
    "source_variables": {
        "albumin": "LBXSAL",
        "creatinine": "LBXSCR",
        "glucose": "LBXGLU",
        "biopro_glucose_not_for_primary_pipeline": "LBXSGL",
        "crp": "LBXHSCRP",
        "lymphocyte_percent": "LBXLYPCT",
        "mcv": "LBXMCVSI",
        "rdw": "LBXRDW",
        "alp": "LBXSAPSI",
        "wbc": "LBXWBCSI",
        "chronological_age": "RIDAGEYR"
    },
    "pipeline": {
        "execution_order": [
            "file_acquisition_and_cycle_tagging",
            "variable_extraction",
            "cross_cycle_bridging_in_native_units",
            "unit_standardization",
            "complete_case_filtering",
            "formula_application_diagnostic_only",
            "validation"
        ],
        "missing_data_policy": "complete_case",
        "imputation_allowed": False,
        "bridge_before_unit_conversion": True,
        "mortality_ingestion_separate_from_biomarker_ingestion": True
    },
    "formula_units": {
        "albumin": "g/L",
        "creatinine": "umol/L",
        "glucose": "mmol/L",
        "crp": "mg/dL before natural log",
        "lymphocyte_percent": "%",
        "mcv": "fL",
        "rdw": "%",
        "alp": "U/L",
        "wbc": "1000 cells/uL",
        "chronological_age": "years"
    },
    "unit_conversions": {
        "albumin": {"operation": "multiply", "factor": 10.0},
        "creatinine": {"operation": "multiply", "factor": 88.4},
        "glucose": {"operation": "multiply", "factor": 0.0555},
        "crp": {"operation": "divide", "factor": 10.0},
        "lymphocyte_percent": None,
        "mcv": None,
        "rdw": None,
        "alp": None,
        "wbc": None
    },
    "bridging": {
        "reference_cycle": "2017_2018",
        "apply_to_cycle": "2015_2016",
        "native_units_required": True,
        "hscrp": {
            "equation": "cobas = 0.8695 * dxc660i + 0.2954",
            "valid_raw_max_mg_L": 23.0,
            "above_range_policy": "flag_and_exclude_from_cross_cycle_comparative_validation"
        },
        "albumin": {
            "equation": "cobas = 0.9581 * dxc660i - 0.0108"
        },
        "creatinine": {
            "equation": "cobas = 0.9515 * dxc660i + 0.06608"
        },
        "alp": {
            "equation": "cobas = 10 ** (0.9986 * log10(dxc660i) + 0.04288)"
        }
    },
    "formula": {
        "xb_coefficients": {
            "intercept": -19.907,
            "albumin": -0.0336,
            "creatinine": 0.0095,
            "glucose": 0.1953,
            "log_crp": 0.0954,
            "lymphocyte_percent": -0.0120,
            "mcv": 0.0268,
            "rdw": 0.3306,
            "alp": 0.00188,
            "wbc": 0.0554,
            "chronological_age": 0.0804
        },
        "mortality_constants": {
            "numerator": 1.51714,
            "denominator": 0.0076927
        },
        "diagnostic_variants": {
            "erratum": {
                "intercept": 141.50,
                "multiplier": -0.00553,
                "denominator": 0.09165,
                "status": "provisional_default_pending_EG-010"
            },
            "supplement": {
                "intercept": 141.50225,
                "multiplier": -0.00553,
                "denominator": 0.090165,
                "status": "sensitivity_variant_pending_EG-010"
            }
        }
    },
    "paths": {
        "raw_data": "data/raw",
        "raw_mortality": "data/raw/mortality/2019_public",
        "interim_data": "data/interim",
        "processed_data": "data/processed",
        "figures": "results/figures",
        "tables": "results/tables",
        "logs": "logs",
        "governance_docs": "docs/governance",
        "methodology_docs": "docs/methodology",
        "paper_reviews": "docs/methodology/paper_reviews",
        "document_staging": "docs/_incoming"
    },
    "documents": {
        "research_protocol": "docs/governance/00_Research_Protocol_v1.0.md",
        "assumption_register": "docs/governance/Assumption_Register.md",
        "decision_log": "docs/governance/Decision_Log.md",
        "evidence_gap_register": "docs/governance/Evidence_Gap_Register.md",
        "evidence_matrix": "docs/governance/Evidence_Matrix.xlsx",
        "literature_matrix": "docs/governance/Literature_Matrix.xlsx",
        "methodology": "docs/methodology/Methodology.md",
        "harmonization_report": "docs/methodology/NHANES_Harmonization_Report.md",
        "replication_protocol": "docs/methodology/Replication_Protocol.md",
        "validation_protocol": "docs/methodology/Validation_Protocol.md",
        "variable_mapping_table": "docs/methodology/Variable_Mapping_Table.xlsx",
        "paper_reviews_directory": "docs/methodology/paper_reviews"
    }
}

config_path = PROJECT_ROOT / "configs" / "agelens_config.json"

if config_path.exists() and not OVERWRITE_CONFIG:
    print(f"Config already exists and was preserved: {config_path}")
    print("To replace it with this updated schema, review the existing file,")
    print("then set OVERWRITE_CONFIG = True and rerun this cell.")
else:
    config_path.write_text(
        json.dumps(CONFIG, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"Config written: {config_path}")


Config already exists and was preserved: <PROJECT_ROOT>\configs\agelens_config.json
To replace it with this updated schema, review the existing file,
then set OVERWRITE_CONFIG = True and rerun this cell.


## 5. Create support files safely

These files are created only when absent.


In [6]:
SUPPORT_FILES = {
    ".gitignore": """# Python
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.venv/
venv/

# Data and generated outputs
data/raw/
data/interim/
data/processed/
results/
logs/

# Local environment
.env
.DS_Store
Thumbs.db
""",
    "requirements.txt": """jupyterlab
numpy
pandas
scipy
matplotlib
pyarrow
pyreadstat
openpyxl
statsmodels
""",
    "README.md": """# AgeLens

Replication-first implementation of Levine Phenotypic Age using NHANES 2015–2016 and 2017–2018 data.

## Governance status

Final scientific results are not authorized while Core Evidence Gaps EG-004, EG-010, and EG-014 remain unresolved or are not explicitly dispositioned by approved Decisions.

## Raw data locations

- `data/raw/2015_2016/`: NHANES 2015–2016 XPT files
- `data/raw/2017_2018/`: NHANES 2017–2018 XPT files
- `data/raw/mortality/2019_public/`: 2019 public-use linked mortality DAT files
- `scripts/`: reproducible R, Python, and shell scripts used by validation or auxiliary workflows
- `docs/governance/`: authoritative governance records and evidence matrices
- `docs/methodology/`: methodology, harmonization, replication, and validation documents
- `docs/methodology/paper_reviews/`: governed paper-review records
- `docs/_incoming/`: optional staging folder used by the setup document installer

Mortality files must be parsed separately using the official NCHS fixed-width specification.

## Notebook order

1. `notebooks/00_setup_agelens.ipynb`
2. biomarker data ingestion
3. preprocessing and harmonization
4. formula diagnostics
5. validation
6. external validation scripts, including `scripts/04_bioage_survey_validation.R`
7. mortality ingestion and outcome analysis only after the replication pipeline is validated
"""
}

for relative_path, content in SUPPORT_FILES.items():
    file_path = PROJECT_ROOT / relative_path
    if file_path.exists() and not OVERWRITE_SUPPORT_FILES:
        print(f"Preserved existing file: {relative_path}")
    else:
        file_path.write_text(content, encoding="utf-8")
        print(f"Wrote: {relative_path}")


Preserved existing file: .gitignore
Preserved existing file: requirements.txt
Preserved existing file: README.md


## 6. Record the execution environment

In [7]:
def package_version(package_name: str) -> str | None:
    try:
        from importlib.metadata import version
        return version(package_name)
    except Exception:
        return None


environment_snapshot = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "working_directory": str(Path.cwd().resolve()),
    "project_root": str(PROJECT_ROOT),
    "packages": {
        name: package_version(name)
        for name in [
            "jupyterlab",
            "numpy",
            "pandas",
            "scipy",
            "matplotlib",
            "pyarrow",
            "pyreadstat",
            "openpyxl",
            "statsmodels",
        ]
    }
}

snapshot_path = PROJECT_ROOT / "logs" / "environment_snapshot.json"
snapshot_path.write_text(
    json.dumps(environment_snapshot, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Environment snapshot written: {snapshot_path}")


Environment snapshot written: <PROJECT_ROOT>\logs\environment_snapshot.json


## 7. Verify setup

The mortality directory is required, but the mortality files themselves are not required for the setup notebook to pass.


In [8]:
required_paths = [
    PROJECT_ROOT / "configs" / "agelens_config.json",
    PROJECT_ROOT / "data" / "raw" / "2015_2016",
    PROJECT_ROOT / "data" / "raw" / "2017_2018",
    PROJECT_ROOT / "data" / "raw" / "mortality" / "2019_public",
    PROJECT_ROOT / "data" / "interim",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "docs" / "_incoming",
    PROJECT_ROOT / "docs" / "governance",
    PROJECT_ROOT / "docs" / "methodology",
    PROJECT_ROOT / "docs" / "methodology" / "paper_reviews",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "scripts",
    PROJECT_ROOT / "src" / "agelens",
    PROJECT_ROOT / "src" / "agelens" / "mortality",
    PROJECT_ROOT / "tests",
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(
        "Setup verification failed. Missing:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

saved_config = json.loads(
    (PROJECT_ROOT / "configs" / "agelens_config.json").read_text(
        encoding="utf-8"
    )
)

assert saved_config["pipeline"]["missing_data_policy"] == "complete_case"
assert saved_config["pipeline"]["imputation_allowed"] is False
assert saved_config["pipeline"]["bridge_before_unit_conversion"] is True
assert saved_config["source_variables"]["glucose"] == "LBXGLU"
assert saved_config["nhanes"]["survey_design"]["weight_variable"] == "WTSAF2YR"
assert saved_config["formula"]["xb_coefficients"]["glucose"] == 0.1953
assert saved_config["project"]["final_scientific_results_allowed"] is False
assert saved_config["mortality"]["format"] == "fixed_width_dat"
assert saved_config["mortality"]["parse_in_setup"] is False

mortality_dir = (
    PROJECT_ROOT / saved_config["mortality"]["raw_directory"]
)

print("✅ AgeLens project setup verified.")
print(f"Project root: {PROJECT_ROOT}")
print(
    "Paper reviews folder: "
    f"{PROJECT_ROOT / 'docs' / 'methodology' / 'paper_reviews'}"
)

canonical_document_status = pd.DataFrame(
    [
        {
            "document": relative_path,
            "present": (
                PROJECT_ROOT / relative_path
            ).exists(),
        }
        for relative_path in DOCUMENT_MANIFEST
    ]
)

print("\nCanonical document status:")
display(canonical_document_status)

missing_canonical_documents = (
    canonical_document_status.loc[
        ~canonical_document_status["present"],
        "document",
    ].tolist()
)

if missing_canonical_documents:
    print(
        "Authoritative documents still missing: "
        f"{len(missing_canonical_documents)}"
    )

    if REQUIRE_ALL_DOCUMENTS:
        raise FileNotFoundError(
            "Canonical document verification failed:\n"
            + "\n".join(missing_canonical_documents)
        )

print(f"Mortality folder: {mortality_dir}")
print("Expected mortality files:")
for cycle, filename in saved_config["mortality"]["files"].items():
    file_path = mortality_dir / filename
    status = "FOUND" if file_path.exists() else "NOT YET PRESENT"
    print(f"  - {cycle}: {filename} [{status}]")


✅ AgeLens project setup verified.
Project root: <PROJECT_ROOT>
Paper reviews folder: <PROJECT_ROOT>\docs\methodology\paper_reviews

Canonical document status:


,document,present
0,docs/governance/00_Research_Protocol_v1.0.md,True
1,docs/governance/Assumption_Register.md,True
2,docs/governance/Decision_Log.md,True
3,docs/governance/Evidence_Gap_Register.md,True
4,docs/governance/Evidence_Matrix.xlsx,True
5,docs/governance/Literature_Matrix.xlsx,True
6,docs/methodology/Methodology.md,True
7,docs/methodology/NHANES_Harmonization_Report.md,True
8,docs/methodology/Replication_Protocol.md,True
9,docs/methodology/Validation_Protocol.md,True


Mortality folder: <PROJECT_ROOT>\data\raw\mortality\2019_public
Expected mortality files:
  - 2015_2016: NHANES_2015_2016_MORT_2019_PUBLIC.dat [FOUND]
  - 2017_2018: NHANES_2017_2018_MORT_2019_PUBLIC.dat [FOUND]
